
# CS452 / Deep Learning — Assignment 1: Facial Affect (CPU-Only)
**Baselines:** ResNet18 & EfficientNet-B0 (multi-task: 8-way expression + 2D valence/arousal).  
**Runs on:** CPU-only 8th‑gen laptop in VS Code.  
**Dataset:** `annotations/*.npy` (e.g., `123_exp.npy`, `123_val.npy`, `123_aro.npy`, `123_lnd.npy`) next to this notebook.  
If images are missing, we render 224×224 landmark heatmaps so the full pipeline still runs.


In [ ]:

# === Environment & Config ===
import os, sys, math, json, time, random, glob, zipfile, pathlib, itertools, statistics
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from torchvision.models import resnet18, efficientnet_b0

from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, average_precision_score, confusion_matrix
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

THIS_DIR = Path.cwd()
CANDIDATES = [
    THIS_DIR/"annotations",
    THIS_DIR/"Dataset"/"Dataset"/"annotations",
    THIS_DIR/"DL_Assignment1_Dataset"/"Dataset"/"Dataset"/"annotations",
    THIS_DIR/"DL_Assignment1_Dataset"/"annotations",
]
ANN_DIR = None
for c in CANDIDATES:
    if c.exists() and any(str(p).endswith("_exp.npy") for p in c.iterdir() if p.is_file()):
        ANN_DIR = c; break
if ANN_DIR is None:
    raise FileNotFoundError("annotations/*.npy not found. Place dataset next to this notebook.")

IMG_DIR = THIS_DIR/"images"  # optional

DEVICE = torch.device("cpu")
NUM_WORKERS = 0
BATCH_SIZE = 32
EPOCHS = 5
VAL_SPLIT = 0.2
USE_LANDMARK_RENDERING_IF_NO_IMAGES = True
IMG_SIZE = 224
USE_PRETRAINED_WEIGHTS = False


In [ ]:

# === Metrics Utils ===

def krippendorff_alpha_nominal(coders):
    '''
    Krippendorff's alpha (nominal). Treat y_true and y_pred (argmax) as two coders.
    coders: list of sequences, each sequence = ratings for N items. Missing values as None.
    '''
    values = set()
    for ratings in coders:
        values.update([r for r in ratings if r is not None])
    values = list(values)
    if not values:
        return float('nan')
    idx = {v:i for i,v in enumerate(values)}
    n = len(values)
    coincidences = np.zeros((n,n), dtype=float)
    N = len(coders[0])
    for i in range(N):
        r = [ratings[i] for ratings in coders if ratings[i] is not None]
        for a in r:
            for b in r:
                coincidences[idx[a], idx[b]] += 1
    Do = 0.0
    for a in range(n):
        for b in range(n):
            if a != b:
                Do += coincidences[a,b]
    m = coincidences.sum(axis=0)
    De = 0.0
    for a in range(n):
        for b in range(n):
            if a != b:
                De += m[a]*m[b]
    if De == 0:
        return 1.0
    return 1.0 - Do/De

def concordance_correlation_coefficient(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true-mean_true)*(y_pred-mean_pred))
    denom = (var_true + var_pred + (mean_true-mean_pred)**2)
    return 1.0 if denom == 0 else (2*cov)/denom

def sagr(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return float(np.mean(np.sign(y_true) == np.sign(y_pred)))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))


In [ ]:

# === Dataset ===

def list_sample_ids(ann_dir: Path):
    ids = sorted(set(p.name.split("_")[0] for p in ann_dir.glob("*_exp.npy")))
    return ids

def load_label_files(ann_dir: Path, sid: str):
    exp = np.load(ann_dir/f"{sid}_exp.npy", allow_pickle=True)
    val = np.load(ann_dir/f"{sid}_val.npy", allow_pickle=True)
    aro = np.load(ann_dir/f"{sid}_aro.npy", allow_pickle=True)
    lnd = np.load(ann_dir/f"{sid}_lnd.npy", allow_pickle=True)
    exp = int(np.array(exp).squeeze())
    val = float(np.array(val).squeeze())
    aro = float(np.array(aro).squeeze())
    lnd = np.array(lnd).reshape(-1, 2)
    return exp, val, aro, lnd

def render_landmark_image(landmarks, size=224):
    img = Image.new("RGB", (size, size), (0,0,0))
    draw = ImageDraw.Draw(img)
    for (x,y) in landmarks:
        xi = int(np.clip(x, 0, size-1)); yi = int(np.clip(y, 0, size-1))
        r = 2
        draw.ellipse((xi-r, yi-r, xi+r, yi+r), fill=(255,255,255))
    return img

class AffectDataset(torch.utils.data.Dataset):
    def __init__(self, ann_dir: Path, img_dir: Path, transform=None):
        self.ann_dir = Path(ann_dir)
        self.img_dir = Path(img_dir) if img_dir is not None else None
        self.transform = transform
        self.ids = list_sample_ids(self.ann_dir)
        self.num_classes = 8

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]
        exp, val, aro, lnd = load_label_files(self.ann_dir, sid)
        img = None
        if self.img_dir is not None:
            for ext in [".jpg",".jpeg",".png",".bmp"]:
                p = self.img_dir/f"{sid}{ext}"
                if p.exists():
                    img = Image.open(p).convert("RGB"); break
        if img is None:
            if USE_LANDMARK_RENDERING_IF_NO_IMAGES:
                img = render_landmark_image(lnd, size=IMG_SIZE)
            else:
                img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0,0,0))
        if self.transform:
            img = self.transform(img)
        target_cls = int(exp)
        target_reg = np.array([val, aro], dtype=np.float32)
        return img, target_cls, target_reg, sid


In [ ]:

# === Data & Loaders ===
from torch.utils.data import random_split

base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_ds = AffectDataset(ANN_DIR, IMG_DIR, transform=None)
n_total = len(full_ds)
n_val = int(VAL_SPLIT * n_total)
n_train = n_total - n_val
train_indices, val_indices = torch.utils.data.random_split(range(n_total), [n_train, n_val], generator=torch.Generator().manual_seed(SEED))

class TransformWrapper(torch.utils.data.Dataset):
    def __init__(self, base_ds, indices, transform):
        self.base = base_ds; self.indices = list(indices); self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        img, y, r, sid = self.base[self.indices[i]]
        img = self.transform(img)
        return img, y, r, sid

train_ds = TransformWrapper(full_ds, train_indices, aug_transform)
val_ds   = TransformWrapper(full_ds, val_indices, base_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Total: {n_total} | Train: {len(train_ds)} | Val: {len(val_ds)}")


In [ ]:

# === Models ===
class MultiTaskModel(nn.Module):
    def __init__(self, backbone_name="resnet18", num_classes=8, use_pretrained=False):
        super().__init__()
        if backbone_name == "resnet18":
            m = resnet18(weights=None if not use_pretrained else torchvision.models.ResNet18_Weights.DEFAULT)
            feat_dim = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m
        elif backbone_name == "efficientnet_b0":
            m = efficientnet_b0(weights=None if not use_pretrained else torchvision.models.EfficientNet_B0_Weights.DEFAULT)
            feat_dim = m.classifier[-1].in_features
            m.classifier[-1] = nn.Identity()
            self.backbone = m
        else:
            raise ValueError("Unsupported backbone")
        self.cls_head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.reg_head = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, 2), nn.Tanh()
        )
    def forward(self, x):
        feats = self.backbone(x)
        logits = self.cls_head(feats)
        reg = self.reg_head(feats)
        return logits, reg

def build_model(backbone):
    model = MultiTaskModel(backbone_name=backbone, num_classes=8, use_pretrained=USE_PRETRAINED_WEIGHTS)
    return model.to(DEVICE)


In [ ]:

# === Train & Eval ===
def train_one_epoch(model, loader, optim, sched=None):
    model.train()
    ce = nn.CrossEntropyLoss()
    regloss = nn.MSELoss()
    losses = []
    for imgs, y_cls, y_reg, _ in loader:
        imgs = imgs.to(DEVICE); y_cls = y_cls.to(DEVICE); y_reg = y_reg.to(DEVICE)
        optim.zero_grad()
        logits, reg = model(imgs)
        loss = ce(logits, y_cls) + regloss(reg, y_reg)
        loss.backward(); optim.step()
        if sched: sched.step()
        losses.append(loss.item())
    return float(np.mean(losses))

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ce = nn.CrossEntropyLoss(); regloss = nn.MSELoss()
    losses = []; y_true_cls=[]; y_pred_cls=[]; y_prob=[]; y_true_val=[]; y_pred_val=[]; y_true_aro=[]; y_pred_aro=[]; sids=[]
    for imgs, y_cls, y_reg, ids in loader:
        imgs = imgs.to(DEVICE)
        logits, reg = model(imgs)
        loss = ce(logits, y_cls.to(DEVICE)) + regloss(reg, y_reg.to(DEVICE))
        losses.append(loss.item())
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(1)
        y_true_cls.extend(y_cls.cpu().numpy().tolist())
        y_pred_cls.extend(preds.tolist())
        y_prob.extend(probs.tolist())
        y_true_val.extend(y_reg[:,0].numpy().tolist())
        y_pred_val.extend(reg[:,0].cpu().numpy().tolist())
        y_true_aro.extend(y_reg[:,1].numpy().tolist())
        y_pred_aro.extend(reg[:,1].cpu().numpy().tolist())
        sids.extend(ids)
    acc = accuracy_score(y_true_cls, y_pred_cls)
    f1  = f1_score(y_true_cls, y_pred_cls, average="macro")
    kappa = cohen_kappa_score(y_true_cls, y_pred_cls)
    alpha = krippendorff_alpha_nominal([y_true_cls, y_pred_cls])
    try:
        auc_macro = roc_auc_score(y_true_cls, np.array(y_prob), multi_class="ovr", average="macro")
        pr_auc_macro = average_precision_score(np.eye(8)[np.array(y_true_cls)], np.array(y_prob), average="macro")
    except Exception:
        auc_macro = float('nan'); pr_auc_macro = float('nan')
    rmse_val = rmse(y_true_val, y_pred_val); rmse_aro = rmse(y_true_aro, y_pred_aro)
    corr_val = pearsonr(y_true_val, y_pred_val)[0] if len(y_true_val)>1 else float('nan')
    corr_aro = pearsonr(y_true_aro, y_pred_aro)[0] if len(y_true_aro)>1 else float('nan')
    sagr_val = sagr(y_true_val, y_pred_val); sagr_aro = sagr(y_true_aro, y_pred_aro)
    ccc_val  = concordance_correlation_coefficient(y_true_val, y_pred_val)
    ccc_aro  = concordance_correlation_coefficient(y_true_aro, y_pred_aro)
    metrics = {
        "loss": float(np.mean(losses)),
        "acc": acc, "f1_macro": f1, "kappa": kappa, "alpha": alpha,
        "auc_macro": auc_macro, "pr_auc_macro": pr_auc_macro,
        "rmse_val": rmse_val, "rmse_aro": rmse_aro,
        "corr_val": float(corr_val), "corr_aro": float(corr_aro),
        "sagr_val": float(sagr_val), "sagr_aro": float(sagr_aro),
        "ccc_val": float(ccc_val), "ccc_aro": float(ccc_aro),
        "y_true_cls": y_true_cls, "y_pred_cls": y_pred_cls, "y_prob": y_prob,
        "y_true_val": y_true_val, "y_pred_val": y_pred_val,
        "y_true_aro": y_true_aro, "y_pred_aro": y_pred_aro, "sids": sids
    }
    return metrics

def run_training(backbone="resnet18"):
    model = build_model(backbone)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, len(train_loader)*EPOCHS))
    hist = {"train_loss": [], "val_loss": [], "val_metrics": []}
    for epoch in range(1, EPOCHS+1):
        t0 = time.time()
        tl = train_one_epoch(model, train_loader, optimizer, scheduler)
        vm = evaluate(model, val_loader)
        hist["train_loss"].append(tl); hist["val_loss"].append(vm["loss"]); hist["val_metrics"].append(vm)
        dt = time.time()-t0
        print(f"[{backbone}] Epoch {epoch}/{EPOCHS} - train_loss={tl:.4f} val_loss={vm['loss']:.4f} acc={vm['acc']:.3f} (t={dt:.1f}s)")
    return model, hist


In [ ]:

# === Train Baselines ===
histories = {}; models = {}
for backbone in ["resnet18", "efficientnet_b0"]:
    model, hist = run_training(backbone)
    models[backbone] = model; histories[backbone] = hist

out_dir = Path("results"); out_dir.mkdir(exist_ok=True)
for b,h in histories.items():
    with open(out_dir/f"{b}_history.json","w") as f:
        json.dump(h, f, indent=2)
print("Saved training histories under ./results/")


In [ ]:

# === Plots & Confusion Matrices ===
def plot_loss(histories):
    plt.figure()
    for b,h in histories.items():
        plt.plot(h["train_loss"], label=f"{b} train")
        plt.plot(h["val_loss"], label=f"{b} val")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training/Validation Loss")
    plt.legend(); plt.show()

def plot_confusion(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))
    plt.figure()
    plt.imshow(cm, interpolation="nearest")
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.colorbar(); plt.show()
    return cm

plot_loss(histories)
for b,h in histories.items():
    vm = h["val_metrics"][-1]
    _ = plot_confusion(vm["y_true_cls"], vm["y_pred_cls"], title=f"{b} — Confusion (Val)")


In [ ]:

# === Qualitative Samples ===
def unnorm(t):
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    return (t*std + mean).clamp(0,1)

def show_samples(backbone, k=12):
    vm = histories[backbone]["val_metrics"][-1]
    y_true = np.array(vm["y_true_cls"]); y_pred = np.array(vm["y_pred_cls"]); sids = vm["sids"]
    tfm = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
                              transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
    correct_idx = np.where(y_true==y_pred)[0][:k]; wrong_idx = np.where(y_true!=y_pred)[0][:k]
    def get_img(sid):
        exp,val,aro,lnd = load_label_files(ANN_DIR, sid)
        img = None
        for ext in [".jpg",".jpeg",".png",".bmp"]:
            p = IMG_DIR/f"{sid}{ext}"
            if p.exists(): img = Image.open(p).convert("RGB"); break
        if img is None: img = render_landmark_image(lnd, size=IMG_SIZE)
        return tfm(img)
    for tag, idxs in [("Correct", correct_idx), ("Incorrect", wrong_idx)]:
        plt.figure()
        n = len(idxs); cols = 4; rows = int(math.ceil(n/cols)) or 1
        for i,ii in enumerate(idxs):
            sid = sids[ii]; img = get_img(sid)
            plt.subplot(rows, cols, i+1); plt.axis("off")
            plt.imshow(unnorm(img).permute(1,2,0).numpy())
            plt.title(f"sid:{sid}\ntrue:{int(y_true[ii])} pred:{int(y_pred[ii])}")
        plt.suptitle(f"{backbone} — {tag} predictions"); plt.show()

for b in ["resnet18", "efficientnet_b0"]:
    show_samples(b, k=8)


In [ ]:

# === Final Metrics Table ===
rows = []
for b,h in histories.items():
    vm = h["val_metrics"][-1]
    rows.append({
        "backbone": b,
        "acc": vm["acc"], "f1_macro": vm["f1_macro"], "kappa": vm["kappa"], "alpha": vm["alpha"],
        "auc_macro": vm["auc_macro"], "pr_auc_macro": vm["pr_auc_macro"],
        "rmse_val": vm["rmse_val"], "rmse_aro": vm["rmse_aro"],
        "corr_val": vm["corr_val"], "corr_aro": vm["corr_aro"],
        "sagr_val": vm["sagr_val"], "sagr_aro": vm["sagr_aro"],
        "ccc_val": vm["ccc_val"], "ccc_aro": vm["ccc_aro"],
    })
df = pd.DataFrame(rows).set_index("backbone")
df
